# Refua Peptide Design Notebook: GLP1R ECD + Exendin-like Binder

> **Audience:** beginner to advanced.  
> **Goal:** create peptide binder candidates for a clinically relevant peptide-druggable receptor context.

## Why this target is practically useful
- **Target:** human **GLP-1 receptor extracellular domain (GLP1R ECD)**.
- **Complex precedent:** PDB **3C59** captures GLP1R ECD bound to an **exendin-4 fragment**.
- **Clinical relevance:** GLP-1 receptor agonist peptides are an established therapeutic class in metabolic disease.

## Workflow
1. Define GLP1R ECD target sequence and a binding-priority region.
2. Use `BinderDesigns.peptide(...)` and `BinderDesigns.disulfide_peptide(...)` helper APIs.
3. Fold one design and inspect generated binder specs for triage.


In [ ]:
%load_ext refua_notebook
import refua_notebook

refua_notebook.activate()


In [ ]:
from refua import BinderDesigns, Complex, Protein


## Biological setup (plain language)

GLP1R is a receptor activated by peptide ligands. In this notebook we design **new peptide candidates** against the ECD binding surface.

| Design input | Choice in this notebook | Why |
|---|---|---|
| Target chain | 3C59 chain A (GLP1R ECD) | Experimentally solved peptide-bound receptor domain |
| Reference ligand | 3C59 chain B exendin-like peptide fragment | Known binder geometry context |
| Binding window | `20..120` on receptor chain A | Broad ECD ligand-facing region for early exploration |


In [ ]:
GLP1R_ECD_3C59 = (
    "RPQGATVSLWETVQKWREYRRQCQRSLTEDPPPATDLFCNRTFDEYACWPDGEPGSFVNVSCPWYLPWASSVPQGHVYRFCTAEGLWLQKDNSSLPWRDLSECEESKRGERSSPEEQLLFLY"
)

# Exendin-like peptide fragment observed in 3C59 chain B.
EXENDIN_FRAGMENT_3C59 = "DLSKQMEEEAVRMFIEWLKNGGPSSGAPPPS"

GLP1R_BINDING_WINDOW = "20..120"


## Build peptide design specs with helper APIs

Helper API presets make it easy to start with reasonable peptide design spaces:
- `BinderDesigns.peptide(length=...)` for a linear candidate family.
- `BinderDesigns.disulfide_peptide(segment_lengths=(...))` for constrained cyclic-like exploration.

We run the linear design first and keep the disulfide setup ready as a follow-up branch.


In [ ]:
target = Protein(
    GLP1R_ECD_3C59,
    ids="A",
    binding_types={"binding": GLP1R_BINDING_WINDOW},
)

linear_peptide = BinderDesigns.peptide(length=len(EXENDIN_FRAGMENT_3C59), ids="P")
disulfide_peptide = BinderDesigns.disulfide_peptide(
    segment_lengths=(8, 6, 5),
    ids="Q",
)

linear_design = Complex([target, linear_peptide], name="glp1r_linear_peptide_design")
disulfide_design = Complex([target, disulfide_peptide], name="glp1r_disulfide_peptide_design")


In [ ]:
{
    "linear_helper_spec": linear_peptide.sequence,
    "disulfide_helper_spec": disulfide_peptide.sequence,
    "reference_bound_peptide_3c59": EXENDIN_FRAGMENT_3C59,
}


## Run one design pass (linear peptide)

`fold()` gives a structure/design hypothesis you can use for ranking and iteration.

> Guardrail: these are computational candidates and should be validated experimentally.


In [ ]:
result_linear = linear_design.fold()

result_linear


In [ ]:
result_linear.binder_specs


In [ ]:
result_linear.chain_design_summary()


## Optional follow-up

Run the disulfide-constrained branch to compare design behavior:

```python
result_disulfide = disulfide_design.fold()
result_disulfide.binder_specs
result_disulfide.chain_design_summary()
```


## Science references

- RCSB PDB 3C59 (GLP1R ECD + exendin-like peptide complex): https://www.rcsb.org/structure/3C59
- Runge et al., *J Biol Chem* (2008), GLP1R ECD ligand-bound crystal structure: https://pubmed.ncbi.nlm.nih.gov/18287102/
- Exenatide insulin secretion study in T2D: https://pubmed.ncbi.nlm.nih.gov/16144950/
- FDA preclinical research basics (why validation is required): https://www.fda.gov/patients/drug-development-process/step-2-preclinical-research
